In [1]:
from processing_data import *

In [2]:
import pickle 
with open('SVM_CT.pkl','rb') as f:
    pre = pickle.load(f)

In [3]:
with open ('Processed_data.pkl','rb') as f:
    df= pickle.load(f)

In [4]:
pre

ColumnTransformer(transformers=[('OHE', OneHotEncoder(),
                                 ['Self_Employed', 'Property_Area', 'Gender']),
                                ('SS', StandardScaler(),
                                 ['ApplicantIncome', 'CoapplicantIncome',
                                  'LoanAmount']),
                                ('FT',
                                 FunctionTransformer(func=<function div_by_30 at 0x000001B65D0EE840>),
                                 ['Loan_Amount_Term']),
                                ('OE', OrdinalEncoder(), ['Married']),
                                ('do_nothing',
                                 FunctionTransformer(func=<function same at 0x000001B65D0EE980>),
                                 ['Credit_History']),
                                ('OE1',
                                 OrdinalEncoder(categories=[['0', '1', '2',
                                                             '3+']]),
                                 ['Dependents']),
                                ('OE2',
                                 OrdinalEncoder(categories=[['Not Graduate',
                                                             'Graduate']]),
                                 ['Education'])])

In [5]:
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,146.412162,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.000000,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.000000,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.000000,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.000000,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.000000,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.000000,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.000000,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.000000,360.0,1.0,Urban,Y


In [6]:
x= df.drop('Loan_Status (Approved)',axis=1)
y= df['Loan_Status (Approved)'].map({'N':0,'Y':1})

In [7]:
from sklearn.model_selection import train_test_split
x_train_raw,x_test_raw,y_train,y_test = train_test_split(x,y,test_size=0.20,random_state=42)

In [8]:
x_train=pre.fit_transform(x_train_raw)
x_test=pre.transform(x_test_raw)

In [9]:
x_train.shape

(491, 15)

In [10]:
y_train.value_counts()

Loan_Status (Approved)
1    342
0    149
Name: count, dtype: int64

In [11]:
!pip install imblearn

Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

In [13]:
oversample= RandomOverSampler(sampling_strategy='minority')
x_over, y_over = oversample.fit_resample(x_train , y_train)
print("Oversampled class distribution:",Counter(y_over))

undersample= RandomUnderSampler(sampling_strategy='majority')
x_under, y_under = undersample.fit_resample(x_train , y_train)
print("Undersampled class distribution:",Counter(y_under))

Oversampled class distribution: Counter({0: 342, 1: 342})
Undersampled class distribution: Counter({0: 149, 1: 149})


In [14]:
from imblearn.over_sampling import SMOTE
x_smote, y_smote = SMOTE().fit_resample(x_train, y_train)
print("SMOTE class distribution:",Counter(y_smote))

SMOTE class distribution: Counter({0: 342, 1: 342})


In [15]:
from sklearn.svm import SVC
svclassifier = SVC(random_state=42)
svclassifier.fit(x_smote,y_smote)

SVC(random_state=42)

In [16]:
import sklearn 
sklearn.set_config(print_changed_only= False)
svclassifier

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='rbf',
    max_iter=-1, probability=False, random_state=42, shrinking=True, tol=0.001,
    verbose=False)

In [17]:
y_pred=svclassifier.predict(x_test)

In [18]:
from sklearn.metrics import accuracy_score , precision_score, recall_score , f1_score , confusion_matrix , classification_report

In [19]:
acc = accuracy_score (y_test,y_pred)
print('Accuracy:',acc)
pr = precision_score (y_test,y_pred)
re = recall_score (y_test,y_pred)
f1 = f1_score (y_test,y_pred)
cm = confusion_matrix (y_test,y_pred)
print('Precisio:',pr,'\nRecall:',re,'\nF1:',f1,'\nConfusion Matrix:',cm)
print('Classification Report:\n',classification_report(y_test,y_pred))

Accuracy: 0.7642276422764228
Precisio: 0.7524752475247525 
Recall: 0.95 
F1: 0.8397790055248618 
Confusion Matrix: [[18 25]
 [ 4 76]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.42      0.55        43
           1       0.75      0.95      0.84        80

    accuracy                           0.76       123
   macro avg       0.79      0.68      0.70       123
weighted avg       0.78      0.76      0.74       123



In [20]:
from sklearn.svm import SVC
svclassifier = SVC(random_state=42)
svclassifier.fit(x_train,y_train)
y_pred=svclassifier.predict(x_test)
from sklearn.metrics import accuracy_score , precision_score, recall_score , f1_score , confusion_matrix , classification_report
acc = accuracy_score (y_test,y_pred)
print('Accuracy:',acc)
pr = precision_score (y_test,y_pred)
re = recall_score (y_test,y_pred)
f1 = f1_score (y_test,y_pred)
cm = confusion_matrix (y_test,y_pred)
print('Precisio:',pr,'\nRecall:',re,'\nF1:',f1,'\nConfusion Matrix:',cm)
print('Classification Report:\n',classification_report(y_test,y_pred))

Accuracy: 0.6504065040650406
Precisio: 0.6504065040650406 
Recall: 1.0 
F1: 0.7881773399014779 
Confusion Matrix: [[ 0 43]
 [ 0 80]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        43
           1       0.65      1.00      0.79        80

    accuracy                           0.65       123
   macro avg       0.33      0.50      0.39       123
weighted avg       0.42      0.65      0.51       123



In [21]:
df.columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area',
       'Loan_Status (Approved)'],
      dtype='object')

In [22]:
import sklearn 
sklearn.set_config(print_changed_only= False)

In [23]:
svclassifier

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='rbf',
    max_iter=-1, probability=False, random_state=42, shrinking=True, tol=0.001,
    verbose=False)

In [24]:
from sklearn.model_selection import GridSearchCV
para_grid = {'C':[0.1,0.5,1,10,200],'gamma':[1,0.1,0.31,0.001],'kernel':['poly','linear','rbf']}
grid=GridSearchCV(SVC(random_state=42),para_grid,scoring='f1',cv=5)
grid.fit(x_smote,y_smote)

GridSearchCV(cv=5, error_score=nan,
             estimator=SVC(C=1.0, break_ties=False, cache_size=200,
                           class_weight=None, coef0=0.0,
                           decision_function_shape='ovr', degree=3,
                           gamma='scale', kernel='rbf', max_iter=-1,
                           probability=False, random_state=42, shrinking=True,
                           tol=0.001, verbose=False),
             n_jobs=None,
             param_grid={'C': [0.1, 0.5, 1, 10, 200],
                         'gamma': [1, 0.1, 0.31, 0.001],
                         'kernel': ['poly', 'linear', 'rbf']},
             pre_dispatch='2*n_jobs', refit=True, return_train_score=False,
             scoring='f1', verbose=0)

In [26]:
grid.best_params_

{'C': 200, 'gamma': 1, 'kernel': 'rbf'}

In [28]:
model = SVC(C=10,gamma=1,random_state=42)
model.fit(x_smote,y_smote)

y_pred = model.predict(x_test)

acc = accuracy_score (y_test,y_pred)
print('Accuracy:',acc)
pr = precision_score (y_test,y_pred)
re = recall_score (y_test,y_pred)
f1 = f1_score (y_test,y_pred)
cm = confusion_matrix (y_test,y_pred)
print('Precisio:',pr,'\nRecall:',re,'\nF1:',f1,'\nConfusion Matrix:',cm)
print('Classification Report:\n',classification_report(y_test,y_pred))

Accuracy: 0.6991869918699187
Precisio: 0.7216494845360825 
Recall: 0.875 
F1: 0.7909604519774011 
Confusion Matrix: [[16 27]
 [10 70]]
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.37      0.46        43
           1       0.72      0.88      0.79        80

    accuracy                           0.70       123
   macro avg       0.67      0.62      0.63       123
weighted avg       0.68      0.70      0.68       123

